## Libraries

In [1]:
import os

In [2]:
run = 'local' #local or cloud
os.environ["APP_ENV"] = run

In [3]:
import sys
sys.path.append('..')

In [4]:
from src.core_grass import*
%matplotlib inline
#%load_ext autoreload
#%autoreload 2

In [5]:
import subprocess
result = subprocess.run(["grass", "--config", "python_path"],
                        check=True, text=True, capture_output=True)
sys.path.append(result.stdout.strip())

# import GRASS Python packages
from collections import defaultdict
import grass.script as gs
import grass.jupyter as gj

# Parameters

In [6]:
#set project variables
Project_Area = 'essex'
projection = '3424' #set to None if you want to use the DEM's original projection
res = '1m'

## Set variables analysis
# source = 'wbd' #wbd or user-defined
# data_scale = 'HUC12' #other options HUC12, HUC10, HUC8 OR the field name of the source data to be split <-- case sensitive
# select_data = '030300020407' #     # Value within the filter_field used for data selection 
# to_headwaters = False #False if only interested in local huc area
# input_shp_name = None #None or must be populated populated with a shapefile polygons if data_scale != HUCXX
# buffer = 2000
# auto_delete = False

In [7]:
## Paths:
parent_dir = pl.Path(os.getcwd()).parent/'data'
grass_data_dir = pl.Path(os.getcwd()).parent/'grassdata'
current_dir = pl.Path(os.getcwd())
outputs_dir =parent_dir/'Rasters'/Project_Area/'Outputs'
vector_dir = parent_dir/'Vectors'/Project_Area
raster_dir = parent_dir/'Rasters'/Project_Area
huc_path = vector_dir/'{}_aoi.geojson'.format(Project_Area)
git_data_repo = current_dir/'data_repository'
# if input_shp_name:
#     input_shp = vector_dir/input_shp_name
directory = outputs_dir

### Start new grass session

In [8]:
# define GRASS data settings (adapt to your needs)
location = Project_Area

In [9]:
# Create a project if it doesn't exist
if os.path.exists(grass_data_dir/location):
    pass
else:
    if len(projection) > 4:
        wkt_text = pyproj.CRS.from_authority("esri", int(projection)).to_wkt()
        with open(vector_dir/'{}_prj.wkt'.format(Project_Area), 'w') as t:
            t.write(wkt_text)
        gs.create_project(path=grass_data_dir,
                      name=location,
                      wkt=vector_dir/'{}_prj.wkt'.format(Project_Area))
    else:
        gs.create_project(path=grass_data_dir,
                  name=location,
                  epsg=projection)

In [10]:
# Start GRASS in default project mapset
session = gj.init(grass_data_dir/location)

#### Review existing files

In [11]:
#list all grass files
grass_maps = list_existing_grass(print_it=True)

Available vector maps:
BloomfieldAve_AOI@PERMANENT
GI_pnts@PERMANENT
GI_pnts_raw@PERMANENT
ParkAve_AOI@PERMANENT
Park_Ave_AOI_row@PERMANENT
ROWLimits_nonParcel@PERMANENT
Roads_ROW@PERMANENT
SOrange_AOI@PERMANENT
bloomfield_raw@PERMANENT
essex_county@PERMANENT
essex_nj_BloomfieldAve_AOI@PERMANENT
essex_nj_ParkAve_AOI@PERMANENT
essex_nj_SOrange_AOI@PERMANENT
inlet_essex_nj@PERMANENT
inlet_essex_nj_ParkAve_AOI@PERMANENT
inlet_essex_nj_ParkAve_AOI_line@PERMANENT
inlet_essex_nj_line@PERMANENT
min_accum_spots_v@PERMANENT
possible_locations_v@PERMANENT
row_v_stream@PERMANENT
row_v_streams@PERMANENT
steep_10_pct_ParkAve_AOI@PERMANENT
stream_essex_nj@PERMANENT
stream_essex_nj_ParkAve_AOI@PERMANENT
stream_essex_nj_ParkAve_AOI_cln@PERMANENT
theshold_points_v@PERMANENT
v_basins_essex_nj@PERMANENT
v_basins_essex_nj_ParkAve_AOI@PERMANENT
v_basins_essex_nj_ParkAve_AOIGI@PERMANENT

Available raster maps:
Annual_NLCD_LndCov_2025_CU_C1V2_dd6131a1_dbf0_4db5_9db9_efe7eb4e8e2f@PERMANENT
MASK@PERMANENT
basi

In [12]:
reg = reg = gs.parse_command('g.region',flags='gp')
cell_size = (float(reg.nsres)+ float(reg.ewres)) /2

In [31]:
threshold = 5000//(cell_size**2) #5000 sf
main_select_data= 'essex_nj'
aoi = 'BloomfieldAve_AOI' #ParkAve_AOI, SOrange_AOI B
select_data = main_select_data+'_'+aoi
gi_raw = 'bloomfield_raw'

In [14]:
#standard variables
dem = 'dem_essex_nj'
c_lines = f'v_carve_{select_data}'
accum = 'row_accum'
drain_dir = 'row_drain_dir'
r_basins = f'r_basins_{select_data}'
v_basins = f'v_basins_{select_data}'
sub_basins = f'subbasins_{select_data}'
v_streams = f'stream_{select_data}'
r_streams = f'r_streams_{select_data}'
r_streams_o = f'r_streams_o_{select_data}'
twi = f'r_twi_{select_data}'
v_outlet = f'outlet_{select_data}'
v_inlet = f'inlet_{select_data}'

In [15]:
dem

'dem_essex_nj'

## Prep Work

In [143]:
gs.run_command('r.mask',flags='r')

In [144]:
gs.run_command('g.region',raster = dem,align=dem,zoom=dem)

In [ ]:
gs.run_command("v.clip", input="ROWLimits_nonParcel", clip=aoi, output=select_data)

In [ ]:
gs.run_command("v.clip", input="roads", clip=aoi, output=f"roads_{select_data}")

In [ ]:
gs.run_command('r.mask',vector = select_data)
gs.run_command('g.region',raster = dem,align=dem,zoom=dem)
reg = gs.parse_command('g.region',raster = dem,align=dem,zoom=dem,flags='gp')

In [19]:
gs.run_command('r.watershed',elevation = dem, threshold=threshold,accumulation = accum, drainage=drain_dir, stream = r_streams,flags='a')

In [20]:
import re
import subprocess

# 1. Start the stream extraction process asynchronously
p = gs.start_command('r.stream.extract',elevation=dem,accumulation=accum,threshold=threshold,stream_vector =v_streams, direction=drain_dir, memory = 3000,stderr=subprocess.PIPE)

# 2. Read the output in real-time
while True:
    line = p.stderr.readline()
    if not line and p.poll() is not None:
        break
    if line:
        # Decode bytes to string and print (or parse for percentages)
        print(line)

# 3. Finalize process check
rc = p.returncode
if rc == 0:
    print("Process completed successfully.")
else:
    print(f"Process failed with return code {rc}.")

Loading input raster maps...



Initializing A* search...



A* Search...



Extracting streams...



Thinning stream segments...



Writing output raster maps...



Writing vector map <stream_essex_nj_BloomfieldAve_AOI>...



Writing attribute data...

Building topology for vector map

<stream_essex_nj_BloomfieldAve_AOI@PERMANENT>...

Registering primitives...



Process completed successfully.


In [21]:
gs.run_command("v.db.renamecolumn", map=v_streams, column="stream_type,strm_type")

In [22]:
gs.run_command("v.clean", input=v_streams, output=v_streams+'_cln', tool="rmline")
gs.run_command('v.out.ogr',input=v_streams+'_cln',output=vector_dir/f'flow_{v_streams}.shp',type= 'line',format='ESRI_Shapefile')
#gs.run_command('v.out.ogr',input=v_streams,output=vector_dir/f'flow_{v_streams}.GeoJSON',type= 'line',format='GeoJSON')

In [23]:
gs.run_command("v.extract", input=v_streams, output=v_inlet+'_line',type = 'line',where = "(strm_type = 'start')") 
gs.run_command("v.to.points",input=v_inlet+'_line', output=v_inlet, use='start')

In [24]:
gs.run_command('r.stream.basins',direction =drain_dir,points =v_inlet,basins=r_basins)
gs.run_command('r.to.vect',input = r_basins,output = v_basins, type='area',column='basin_col', flags='s')

In [25]:
gs.run_command('v.out.ogr',input=v_basins,output=vector_dir/f'TDA_less_than_5000_{aoi}.shp',type= 'area',format='ESRI_Shapefile')

In [26]:
aoi

'BloomfieldAve_AOI'

### SLOPE

In [27]:
gs.run_command('r.slope.direction',elevation = dem,direction =drain_dir,dir_type ='auto',slope_measure  ='percent',steps =7, output ='slope_dir',flags='a')

In [28]:
gs.mapcalc("steep_10_pct = if({0} > 0.1, 1, null())".format('slope_dir'))

In [29]:
gs.run_command('r.to.vect',input = 'steep_10_pct',output = f'steep_10_pct_{aoi}', type='area',column='steep_col', flags='s')

In [30]:
gs.run_command('v.out.ogr',input=f'steep_10_pct_{aoi}',output=vector_dir/f'slope_more_than_10_pct_{aoi}.shp',type= 'area',format='ESRI_Shapefile')

## Work for GI Screening

In [ ]:
#clean
gs.run_command("v.multi2singlepart", input=gi_raw, output=gi_raw.replace('raw','exp'))
gs.run_command("v.clean", input=gi_raw.replace('raw','exp'),output=gi_raw.replace('raw','cln'), tool='rmdangle,rmbridge,bpol,rmarea',threshold='10,20,0,5',flags='c')

In [47]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cln'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cln')}.shp',type= 'area',format='ESRI_Shapefile')

In [50]:
#remove steep areas
gs.run_command("v.overlay", ainput=gi_raw.replace('raw','cln'), binput=f'steep_10_pct_{aoi}', operator="not", output=gi_raw.replace('raw','ces'),flags='t')

In [51]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','ces'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','ces')}.shp',type= 'area',format='ESRI_Shapefile')

In [52]:
#remove less than 5000 sf areas
gs.run_command("v.overlay", ainput=gi_raw.replace('raw','ces'), binput=v_basins, operator="not", output=gi_raw.replace('raw','cesd'),flags='t')

In [53]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cesd'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cesd')}.shp',type= 'area',format='ESRI_Shapefile')

In [61]:
#separate by flow path
gs.run_command('v.buffer',input=v_streams+'_cln',output=v_streams+'_cln'+'_buffer',distance = 1)
gs.run_command("v.overlay", ainput=gi_raw.replace('raw','cesd'), binput=v_streams+'_cln'+'_buffer', operator="not", output=gi_raw.replace('raw','cesdf'),flags='t')

In [62]:
#clean
gs.run_command("v.multi2singlepart", input=gi_raw.replace('raw','cesdf'), output=gi_raw.replace('raw','cesdfe'))
gs.run_command("v.clean", input=gi_raw.replace('raw','cesdfe'),output=gi_raw.replace('raw','cesdfec'), tool='rmdangle,rmbridge,bpol,rmarea',threshold='10,20,0,5',flags='c')

In [63]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cesdfec'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdfec')}.shp',type= 'area',format='ESRI_Shapefile')

In [65]:
#select down
gs.run_command('v.buffer',input=v_streams+'_cln',output=v_streams+'_cln'+'select_buffer',distance = 8)
gs.run_command('v.buffer',input=f"roads_{select_data}",output=f"roads_{select_data}"+'select_buffer',distance = 8)

In [67]:
gs.run_command("v.select", ainput=gi_raw.replace('raw','cesdfec'), binput=v_streams+'_cln'+'select_buffer', output=gi_raw.replace('raw','cesdfec8f'), operator="intersects",flags='t')
gs.run_command("v.select", ainput=gi_raw.replace('raw','cesdfec8f'), binput=f"roads_{select_data}"+'select_buffer', output=gi_raw.replace('raw','cesdfec8f8r'), operator="intersects",flags='tr')

In [113]:
gs.run_command('v.db.addtable',map=gi_raw.replace('raw','cesdfec8f8r'))

In [134]:
#get raster info
gs.run_command("v.rast.stats", map=gi_raw.replace('raw','cesdfec8f8r'), raster="TCC", column_prefix="TCC_Pct",method ='average',flags='cd')

In [135]:
gs.run_command("v.rast.stats", map=gi_raw.replace('raw','cesdfec8f8r'), raster="EC_CST", column_prefix="EC_CST_Pct",method ='average',flags='cd')

In [136]:
gs.run_command('v.out.ogr',input=gi_raw.replace('raw','cesdfec8f8r'),output=vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdfec8f8r')}.geojson',type= 'area',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdfec8f8r')}.geojson').to_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdfec8f8r')}.shp')

/tmp/ipykernel_248/249849880.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gpd.read_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdfec8f8r')}.geojson').to_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdfec8f8r')}.shp')
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Pct_average' to 'Pct_averag'
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'TCC_Pct_average' to 'TCC_Pct_av'
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'EC_CST_Pct_average' to 'EC_CST_Pct'
  ogr_write(


## WORK FOR GI AREAS Attribution

In [139]:
# Load or create your GeoDataFrame
gdf_cent = gpd.read_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdfec8f8r')}.geojson')

# Create a new column with centroid points
gdf_cent["geometry"] = gdf_cent.geometry.centroid
gdf_cent["basin_col"] = gdf_cent['cat']

gdf_cent.to_file(vector_dir/'GI_Opps_pnts.shp')

/tmp/ipykernel_248/1449311550.py:8: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_cent.to_file(vector_dir/'GI_Opps_pnts.shp')
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Pct_average' to 'Pct_averag'
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'TCC_Pct_average' to 'TCC_Pct_av'
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'EC_CST_Pct_average' to 'EC_CST_Pct'
  ogr_write(


In [140]:
gdf_cent

,cat,Pct_average,TCC_Pct_average,EC_CST_Pct_average,geometry,basin_col
0,44,NaN,0.0,12.000000,POINT (582015.409 701476.111),44
1,53,NaN,0.0,14.000000,POINT (581604.494 702176.166),53
2,55,NaN,NaN,14.000000,POINT (581465.698 702449.844),55
3,56,NaN,0.0,18.000000,POINT (581393.003 702948.573),56
4,67,0.0,0.0,17.006329,POINT (580126.925 705160.724),67
...,...,...,...,...,...,...
60,463,0.0,0.0,14.000000,POINT (582395.335 700816.507),463
61,464,0.0,0.0,14.000000,POINT (582600.828 700640.157),464
62,466,0.0,0.0,18.000000,POINT (579972.954 705554.6),466
63,467,NaN,0.0,14.000000,POINT (579561.865 706204.48),467


In [141]:
gs.run_command('v.import', input= vector_dir/'GI_Opps_pnts.shp', output= 'GI_pnts_raw')

In [ ]:
gs.run_command('r.stream.snap',input = 'GI_pnts_raw', output='GI_pnts',stream_rast = r_streams,radius = 20/cell_size)

In [ ]:
gs.run_command('v.db.connect', map='GI_pnts',table='GI_pnts_raw',overwrite =True)

In [ ]:
gs.run_command('r.stream.basins',points = 'GI_pnts', direction=drain_dir,basins=r_basins+'GI')

In [ ]:
gs.run_command('r.to.vect',input = r_basins+'GI',output = v_basins+'GI', type='area',column='basin_col', flags='s')

In [ ]:
gs.run_command('v.out.ogr',input=v_basins+'GI',output=vector_dir/f'{v_basins}_GI.geojson',type= 'area',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{v_basins}_GI.geojson').to_file(vector_dir/f'{v_basins}_GI.shp')

In [ ]:
basin_data = gpd.read_file(vector_dir/f'{v_basins}_GI.shp')
basin_data.set_index('basin_col',inplace=True)
#attach shape info
basin_data['TDA_sf'] = basin_data.geometry.area
basin_data['TDA_acre'] = basin_data.geometry.area/43560
tda_ref = basin_data[['TDA_sf']].to_dict()

In [ ]:
GI_data = gpd.read_file(vector_dir/f'{aoi}_{gi_raw.replace('raw','cesdfec8f8r')}.shp')
#attach shape info
GI_data['Area_sf'] = GI_data.geometry.area
GI_data['Perim_ft'] = GI_data.geometry.length
GI_data['Width_ft'] = (2*GI_data.geometry.area)/GI_data.geometry.length
GI_data['Length_ft'] = GI_data.geometry.area/GI_data['Width_ft']
GI_data['LR'] = GI_data.apply(lambda x: (tda_ref['TDA_sf'][x.basin_col]/x.Area_sf) if x.basin_col in tda_ref['TDA_sf'].keys() else 0,axis=1)

In [ ]:
gdf_cent.set_index('basin_col',inplace=True)

In [ ]:
#lat long

In [107]:
gdf_cent['Lat'] = 

,cat,Area,Width,Perim,LR,geometry
basin_col,,,,,,
1,53,464.773,7.10564,130.8180,0.043447,POINT (581604.494 702176.166)
2,55,592.563,8.91066,133.0010,1.231330,POINT (581465.698 702449.844)
3,56,611.323,9.57601,127.6780,31.725800,POINT (581393.003 702948.573)
4,67,1653.040,11.00820,300.3300,7.020760,POINT (580126.925 705160.724)
5,70,2382.360,9.65758,493.3660,8.982310,POINT (579936.686 705512.057)
6,89,1201.360,7.59537,316.3400,NaN,POINT (579042.832 707109.939)
7,130,643.515,9.42244,136.5920,12.030800,POINT (580264.417 704832.893)
8,182,427.819,8.11839,105.3950,2.460690,POINT (581312.479 702739.258)
9,233,1973.470,12.99190,303.7990,0.506833,POINT (580605.637 704242.143)


In [38]:
gdf = gpd.read_file(vector_dir/'bloomfield_raw.shp')

In [39]:
gdf['geometry'] = gdf['geometry'].buffer(-2).buffer(2)

In [40]:
gdf.to_file(vector_dir/'GI_Opps_refined_geo.shp')

In [49]:
flow_paths = gpd.read_file(vector_dir/f'flow_{v_streams}.shp') 

In [55]:
flow_paths_buffer = flow_paths.copy()
flow_paths_buffer['geometry'] = flow_paths.buffer(distance=2)

In [56]:
gdf_split = gpd.overlay(gdf, flow_paths_buffer, how="difference", keep_geom_type=True)

In [58]:
gdf_split_exp = gdf_split.explode(ignore_index=True)

In [56]:
gdf_split_exp.to_file(vector_dir/f'{gi_raw.replace('raw','cesdfc')}.shp')

NameError: name 'gdf_split_exp' is not defined

In [30]:
#gs.run_command('r.stream.basins',direction =drain_dir,stream_rast =r_streams,basins=r_basins)


In [22]:
#gs.mapcalc("theshold_points = if({0} <= ({1} + 8) & !isnull({2}),1, null())".format(accum, threshold,r_streams))

In [ ]:
#gs.run_command('r.to.vect',input = r_basins,output = v_basins, type='area',column='basin_col', flags='s')

In [42]:
#gs.run_command('r.stream.order',elevation=demstream_rast=r_streams, direction=drain_dir, accumulation =accum,stream_vect =v_streams,strahler =r_streams_o)

In [43]:
#gs.run_command('r.stream.order',stream_rast=r_streams, direction=drain_dir,strahler =r_streams_o)

In [63]:
v_basins{aoi}

SyntaxError: unmatched '}' (292614801.py, line 1)

In [17]:
cell_size = (float(reg.nsres)+ float(reg.ewres)) /2

NameError: name 'reg' is not defined

## Watershed Delineation

In [22]:
#convert lines to points
from shapely.geometry import Point

gdf = gpd.read_file(vector_dir/'pipes.shp')
# Assuming 'gdf' is your existing GeoDataFrame with LineString geometries
gdf['geometry'] = gdf['geometry'].apply(lambda line: Point(line.coords[0]))

gdf.to_file(outlet_point)

/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field INSTALLYEA created as String field, though DateTime requested.
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field LASTINSPEC created as String field, though DateTime requested.
  ogr_write(


In [23]:
#import outlet
gs.run_command('v.import', input= outlet_point,extent='region', output= v_outlet)

In [24]:
gs.run_command('r.stream.snap',input = v_outlet, output=f'{v_outlet}_snap',stream_rast = r_streams,radius = 3)

In [25]:
gs.run_command('v.db.connect', map=f'{v_outlet}_snap',table=v_outlet)

In [27]:
# query raster cells (a new column will be added to existing table)
gs.run_command('v.what.rast', map=f'{v_outlet}_snap', raster=accum, column='daa')

In [28]:
gs.run_command('v.out.ogr',input=f'{v_outlet}_snap',output=vector_dir/f'{v_outlet}_snap.geojson',type= 'point',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{v_outlet}_snap.geojson').to_file(vector_dir/f'{v_outlet}_snap.shp')

/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field INSTALLYEA created as String field, though DateTime requested.
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field LASTINSPEC created as String field, though DateTime requested.
  ogr_write(


In [29]:
gs.run_command('r.stream.basins',points = f'{v_outlet}_snap', direction=drain_dir,basins=r_basins)

KeyboardInterrupt: 

In [73]:
gs.run_command('r.to.vect',input = r_basins,output = v_basins, type='area',column='basin_col', flags='s')

In [74]:
gs.run_command('v.out.ogr',input=v_basins,output=vector_dir/f'{v_basins}.geojson',type= 'area',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{v_basins}.geojson').to_file(vector_dir/f'{v_basins}.shp')

In [76]:
m = gj.InteractiveMap(height=600,width=1200)
m.add_vector(v_outlet)
m.add_vector(f'{v_outlet}_snap')
m.add_vector(v_basins)
m.show()

Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

## END